Week 12 · Day 4 — Full Transformer Forward Pass
Why this matters

Today we stitch everything together: encoder + decoder + positional encodings.
This is the first time you’ll see the Transformer architecture as a whole, ready for training on seq2seq tasks (translation, summarization, etc.).

Theory Essentials

Encoder: stack of blocks → encodes source sequence.

Decoder: stack of blocks → generates target sequence step by step.

Positional Encoding: added to embeddings (since no recurrence/convolution).

Final Linear + Softmax: map decoder outputs to vocabulary probabilities.

Core pipeline:

Input tokens → Embedding + PE → Encoder → context

Target tokens → Embedding + PE → Decoder (with mask + context) → predictions

In [1]:
# Setup
import torch, torch.nn as nn
torch.manual_seed(42)

# Positional Encoding
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * -(torch.log(torch.tensor(10000.0)) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0)) # [1, max_len, d_model]

    def forward(self, x):
        return x + self.pe[:, :x.size(1), :]

# Encoder Block (from Day 2)
class EncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        attn_out, _ = self.attn(x, x, x)
        x = self.norm1(x + attn_out)
        ff_out = self.ff(x)
        return self.norm2(x + ff_out)

# Decoder Block (from Day 3)
class DecoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.norm1, self.norm2, self.norm3 = nn.LayerNorm(d_model), nn.LayerNorm(d_model), nn.LayerNorm(d_model)
        self.ff = nn.Sequential(
            nn.Linear(d_model, d_ff), nn.ReLU(), nn.Linear(d_ff, d_model)
        )
    def forward(self, x, enc_out, tgt_mask=None):
        attn_out, _ = self.self_attn(x, x, x, attn_mask=tgt_mask)
        x = self.norm1(x + attn_out)
        attn_out, _ = self.cross_attn(x, enc_out, enc_out)
        x = self.norm2(x + attn_out)
        ff_out = self.ff(x)
        return self.norm3(x + ff_out)

# Full Transformer
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=32, num_heads=2, d_ff=64, num_layers=2, max_len=50):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.pe = PositionalEncoding(d_model, max_len)

        self.enc_layers = nn.ModuleList([EncoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.dec_layers = nn.ModuleList([DecoderBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])

        self.fc_out = nn.Linear(d_model, tgt_vocab)

    def forward(self, src, tgt, tgt_mask=None):
        # src: [batch, src_len], tgt: [batch, tgt_len]
        src = self.pe(self.src_emb(src))
        tgt = self.pe(self.tgt_emb(tgt))

        for layer in self.enc_layers:
            src = layer(src)
        enc_out = src

        for layer in self.dec_layers:
            tgt = layer(tgt, enc_out, tgt_mask)

        return self.fc_out(tgt)

# Example: vocab of 20 tokens
src = torch.randint(0, 20, (2, 5))
tgt = torch.randint(0, 20, (2, 5))

# causal mask
tgt_len = tgt.size(1)
mask = torch.triu(torch.ones(tgt_len, tgt_len), diagonal=1).masked_fill(torch.triu(torch.ones(tgt_len, tgt_len),1)==1, float("-inf"))

model = Transformer(src_vocab=20, tgt_vocab=20)
out = model(src, tgt, tgt_mask=mask)
print("Output shape:", out.shape)  # [batch, tgt_len, vocab]


Output shape: torch.Size([2, 5, 20])


1) Core (10–15 min)
Task: Run the Transformer with num_layers=1 vs num_layers=3. Check how the forward pass still preserves shape.

In [2]:
for layers in [1,3]:
    model = Transformer(20,20,num_layers=layers)
    print(f"Layers={layers}, Output shape:", model(src,tgt,mask).shape)


Layers=1, Output shape: torch.Size([2, 5, 20])
Layers=3, Output shape: torch.Size([2, 5, 20])


2) Practice (10–15 min)
Task: Change vocab size from 20 → 100 and rerun.

In [3]:
model = Transformer(src_vocab=100, tgt_vocab=100)
out = model(src % 100, tgt % 100, mask)
print("New vocab Output shape:", out.shape)


New vocab Output shape: torch.Size([2, 5, 100])


3) Stretch (optional, 10–15 min)
Task: Print the predicted token IDs (argmax across vocab).

In [4]:
pred_ids = out.argmax(dim=-1)
print("Predicted token IDs:\n", pred_ids)


Predicted token IDs:
 tensor([[88, 88, 78, 55, 78],
        [77, 28, 46, 81, 88]])


Yes ✅ exactly — what you’re printing (pred_ids = out.argmax(dim=-1)) are the IDs of the tokens with the highest probability at each target position.

Here’s what happened under the hood in your code:

Your model outputs out with shape [batch, tgt_len, vocab].

For each token position, you get a vector of logits (scores) across the vocabulary.

argmax(dim=-1) picks the index of the highest logit → the token ID with the max probability.

Those IDs are what you printed (tensor([[88, 88, 78, 55, 78], ...])).

They’re not words yet — just integers.

If you had a tokenizer/dictionary, you could map them back to words.

Mini-Challenge (≤40 min)

Build a Tiny Translation Simulator.

Define a toy “source vocab” of numbers 0–9.

Define target vocab = reversed numbers.

Run forward pass with your Transformer.

Print predictions vs expected targets.

Acceptance Criteria:

Forward pass runs end-to-end.

Predictions can be decoded into token IDs.

Short explanation: how encoder → decoder → vocab mapping works.

In [5]:
# Setup
import torch, torch.nn as nn
torch.manual_seed(0)

# ---- Positional Encoding ----
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=256):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-torch.log(torch.tensor(10000.0))/d_model))
        pe[:,0::2] = torch.sin(pos*div); pe[:,1::2] = torch.cos(pos*div)
        self.register_buffer("pe", pe.unsqueeze(0))  # [1, max_len, d_model]
    def forward(self, x):  # x: [B,T,D]
        return x + self.pe[:, :x.size(1), :]

# ---- Encoder/Decoder Blocks ----
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model,d_ff), nn.ReLU(), nn.Linear(d_ff,d_model))
        self.norm2 = nn.LayerNorm(d_model)
    def forward(self, x):
        a,_ = self.attn(x,x,x); x = self.norm1(x+a)
        f = self.ff(x);         x = self.norm2(x+f)
        return x

class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff):
        super().__init__()
        self.self_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.cross_attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.norm1 = nn.LayerNorm(d_model); self.norm2 = nn.LayerNorm(d_model); self.norm3 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model,d_ff), nn.ReLU(), nn.Linear(d_ff,d_model))
    def forward(self, x, enc_out, tgt_mask=None):
        a,_ = self.self_attn(x,x,x, attn_mask=tgt_mask); x = self.norm1(x+a)
        a,_ = self.cross_attn(x, enc_out, enc_out);      x = self.norm2(x+a)
        f = self.ff(x);                                   x = self.norm3(x+f)
        return x

# ---- Full Tiny Transformer ----
class Transformer(nn.Module):
    def __init__(self, src_vocab, tgt_vocab, d_model=32, n_heads=2, d_ff=64, n_layers=2, max_len=64):
        super().__init__()
        self.src_emb = nn.Embedding(src_vocab, d_model)
        self.tgt_emb = nn.Embedding(tgt_vocab, d_model)
        self.pe = PositionalEncoding(d_model, max_len)
        self.enc = nn.ModuleList([EncoderBlock(d_model,n_heads,d_ff) for _ in range(n_layers)])
        self.dec = nn.ModuleList([DecoderBlock(d_model,n_heads,d_ff) for _ in range(n_layers)])
        self.fc_out = nn.Linear(d_model, tgt_vocab)
    def forward(self, src, tgt_in, tgt_mask=None):
        src = self.pe(self.src_emb(src))
        tgt = self.pe(self.tgt_emb(tgt_in))
        for layer in self.enc: src = layer(src)
        enc_out = src
        for layer in self.dec: tgt = layer(tgt, enc_out, tgt_mask)
        return self.fc_out(tgt)   # [B, T_tgt, V_tgt]

# ---- Toy vocab ----
# digits 0..9 plus special tokens
PAD, SOS, EOS = 10, 11, 12
V = 13  # vocab size

id2tok = {**{i:str(i) for i in range(10)}, PAD:"<pad>", SOS:"<sos>", EOS:"<eos>"}
tok2id = {v:k for k,v in id2tok.items()}

def decode(ids): return [id2tok[i] for i in ids]

# ---- Create one batch: source sequence & "reversed" target ----
# Example source: [1, 9, 3, 4]
src_seq = torch.tensor([[1,9,3,4]])               # [B=1, T_src=4]
tgt_gold = torch.tensor([[4,3,9,1,EOS]])          # golden translation (reversed + EOS)

# Teacher-forcing inputs: start with SOS, then gold[:-1]
tgt_in = torch.tensor([[SOS,4,3,9,1]])            # [B=1, T_tgt=5]

# ---- Causal mask for decoder self-attention ----
T = tgt_in.size(1)
tgt_mask = torch.triu(torch.full((T,T), float('-inf')), diagonal=1)

# ---- Run model ----
model = Transformer(src_vocab=V, tgt_vocab=V)
logits = model(src_seq, tgt_in, tgt_mask)         # [1, 5, V]

# ---- Decode predictions (greedy) ----
pred_ids = logits.argmax(dim=-1)[0].tolist()      # [5]
gold_ids = tgt_gold[0].tolist()                   # [5]

print("SRC:        ", decode(src_seq[0].tolist()))
print("TGT_IN:     ", decode(tgt_in[0].tolist()))
print("PRED_IDS:   ", pred_ids, "→", decode(pred_ids))
print("GOLD_IDS:   ", gold_ids, "→", decode(gold_ids))


SRC:         ['1', '9', '3', '4']
TGT_IN:      ['<sos>', '4', '3', '9', '1']
PRED_IDS:    [0, 12, 7, 0, 0] → ['0', '<eos>', '7', '0', '0']
GOLD_IDS:    [4, 3, 9, 1, 12] → ['4', '3', '9', '1', '<eos>']




**`SRC:`**

```
['1', '9', '3', '4']
```

* This is your **source sequence** (the “sentence” in the source language).
* Here it’s digits `1,9,3,4` — the input to the encoder.

---

**`TGT_IN:`**

```
['<sos>', '4', '3', '9', '1']
```

* This is the **decoder input sequence**.
* Starts with `<sos>` (start-of-sequence).
* Then you feed in the gold target tokens shifted right → `[4,3,9,1]`.
* That way the model learns to predict the **next** token at each step.

---

**`PRED_IDS:`**

```
[0, 12, 7, 0, 0] → ['0', '<eos>', '7', '0', '0']
```

* These are the model’s predicted token IDs (via `argmax`).
* They’re decoded into your toy vocabulary.
* Since the model is untrained/random, the predictions don’t match the task (it guessed tokens like `"0"` or `<eos>` early).

---

**`GOLD_IDS:`**

```
[4, 3, 9, 1, 12] → ['4', '3', '9', '1', '<eos>']
```

* This is the **expected target output** (the true “translation” for your task).
* It’s just the source digits reversed + `<eos>`.
* So from `[1,9,3,4]` → `[4,3,9,1,<eos>]`.

---

### 🔑 Big Picture

* **Encoder**: processed `[1,9,3,4]`.
* **Decoder input (TGT\_IN)**: gave it `<sos>` + shifted gold sequence.
* **Model predictions (PRED\_IDS)**: tried to guess the next tokens → currently random, since no training.
* **Gold sequence (GOLD\_IDS)**: the ground-truth reversed numbers.

👉 The important part: you confirmed that the whole pipeline (embedding → encoder → decoder → vocab projection → argmax) works end-to-end.



Notes / Key Takeaways

Transformer = Encoder stack + Decoder stack + PE + final softmax.

Encoder builds contextual source representations.

Decoder generates autoregressively, attending to encoder outputs.

Masking prevents “future leaks.”

Shape: [batch, tgt_len, vocab].

Architecture is highly parallelizable vs RNNs.

Reflection

Why do we add positional encodings both to source and target embeddings?

What does the final linear + softmax layer represent in practice?

1. Why do we add positional encodings both to source and target embeddings?

The encoder and decoder both use self-attention, which is order-agnostic (it just sees a set of vectors).

Without position info, "dog bites man" = "man bites dog".

So we add positional encodings:

Source side: so the encoder knows the order of input tokens.

Target side: so the decoder knows the order of tokens it has generated so far (important for masked self-attention).

2. What does the final linear + softmax layer represent in practice?

The decoder outputs a vector of size d_model for each token position.

The final linear layer projects this vector into the vocabulary space [vocab_size].

The softmax turns those scores (logits) into probabilities → “likelihood of each token being the next one.”

In practice, this is how the model decides: given all context, what word/token should I write next?

👉 Memory hook:

Positional encodings = give the model a sense of order.

Linear + softmax = turn context into an actual next-token choice.